# AdS4 Phase-Lock Probe-Break Test (v8.1)

This notebook tests where constraint stacking fails using **structured probe corruption** rather than arbitrary random noise.

We compare:
- full stack
- dropped GEO probe
- corrupted GEO probe

The corruption is applied directly to the GEO observable, so the break test is easier to interpret physically.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1 / (f_true(r) + 1e-6)

def geo_true(r):
    return torch.sqrt(f_true(r) + 0.7 * g_true(r))


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return F.softplus(self.net(x))

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()

    def forward(self, r):
        return self.f(r), self.g(r)


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_true(r)
    if corruption_strength == 0.0:
        return base
    # Structured corruption: low-frequency bias + mild oscillation over the radial domain.
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return base * radial_bias * oscillation

def train(mode='full', epochs=1000, corruption_strength=0.12):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    loss_hist = []
    consistency_hist = []

    geo_target = geo_true(r) if mode != 'corrupt_geo' else structured_geo_target(r, corruption_strength)

    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        # Base metric supervision
        loss_f = ((f - f_true(r))**2).mean()
        loss_g = ((g - g_true(r))**2).mean()

        # GEO probe term
        geo_pred = torch.sqrt(f + 0.7 * g + 1e-6)
        if mode == 'drop_geo':
            loss_geo = torch.tensor(0.0, device=device)
        else:
            loss_geo = ((geo_pred - geo_target)**2).mean()

        # AdS-style consistency term
        loss_consistency = ((f * g - 1.0)**2).mean()

        loss = loss_f + loss_g + loss_geo + 0.15 * loss_consistency
        loss.backward()
        opt.step()

        loss_hist.append(loss.item())
        consistency_hist.append(loss_consistency.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        geo_pred = torch.sqrt(f_pred + 0.7 * g_pred + 1e-6)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2

    return {
        'model': m,
        'loss_hist': loss_hist,
        'consistency_hist': consistency_hist,
        'f_pred': f_pred,
        'g_pred': g_pred,
        'geo_pred': geo_pred,
        'geo_target': geo_target,
        'metric_err': metric_err,
    }


In [ ]:
run_full = train(mode='full')
run_drop = train(mode='drop_geo')
run_corrupt = train(mode='corrupt_geo', corruption_strength=0.12)


In [ ]:
# Loss comparison
plt.figure(figsize=(8, 4))
plt.plot(run_full['loss_hist'], label='full stack')
plt.plot(run_drop['loss_hist'], label='drop GEO')
plt.plot(run_corrupt['loss_hist'], label='corrupt GEO')
plt.legend()
plt.title('probe-break loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.tight_layout()
plt.show()


In [ ]:
# Main diagnostic figure
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

axes[0].plot(r.cpu(), f_true(r).cpu(), label='f true')
axes[0].plot(r.cpu(), run_full['f_pred'].cpu(), '--', label='f full')
axes[0].plot(r.cpu(), run_drop['f_pred'].cpu(), ':', label='f drop GEO')
axes[0].set_title('f(r): degeneracy return')
axes[0].legend(fontsize=8)

axes[1].plot(r.cpu(), g_true(r).cpu(), label='g true')
axes[1].plot(r.cpu(), run_full['g_pred'].cpu(), '--', label='g full')
axes[1].plot(r.cpu(), run_drop['g_pred'].cpu(), ':', label='g drop GEO')
axes[1].set_title('g(r): probe removal')
axes[1].legend(fontsize=8)

axes[2].plot(run_full['loss_hist'], label='full')
axes[2].plot(run_drop['loss_hist'], label='drop GEO')
axes[2].plot(run_corrupt['loss_hist'], label='corrupt GEO')
axes[2].set_title('stack break threshold')
axes[2].legend(fontsize=8)

axes[3].plot(r.cpu(), geo_true(r).cpu(), label='GEO true')
axes[3].plot(r.cpu(), run_full['geo_pred'].cpu(), '--', label='full GEO pred')
axes[3].plot(r.cpu(), run_corrupt['geo_target'].cpu(), ':', label='corrupt GEO target')
axes[3].set_title('structured probe corruption')
axes[3].legend(fontsize=8)

axes[4].plot(r.cpu(), run_full['metric_err'].cpu(), label='full metric err')
axes[4].plot(r.cpu(), run_drop['metric_err'].cpu(), label='drop GEO err')
axes[4].plot(r.cpu(), run_corrupt['metric_err'].cpu(), label='corrupt GEO err')
axes[4].set_title('error widening')
axes[4].legend(fontsize=8)

axes[5].plot(run_full['consistency_hist'], label='full consistency')
axes[5].plot(run_drop['consistency_hist'], label='drop GEO consistency')
axes[5].plot(run_corrupt['consistency_hist'], label='corrupt GEO consistency')
axes[5].set_title('consistency drift')
axes[5].legend(fontsize=8)

plt.suptitle('AdS4 Phase-Lock Probe-Break Test (v8.1)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v8_1_probe_break.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v8_1_probe_break.png')
